# traintai — one training round (Colab, TPU v5e-1)

Runs one full training round of the tai NPC dialog model end to end via
`src/round.py`: **prepare → SFT top-up → GRPO → forge → sim_eval**, starting
from the ship checkpoint `ple-st-r16-grpo.pt` (honest forge pass 74%).

**Before you start: Runtime → Change runtime type → TPU (v5e-1).**
Every round-path script auto-detects the TPU through `src/device.py`
(XLA TPU → CUDA → MPS → CPU), so no code flags are needed.

Round state (see `AGENTS.md` for the measured basis):

- `r16` is the ship checkpoint. Rounds `r17`–`r22` (GRPO action-reward
  shaping, four variants) are all measured **non-ship** — reward shaping on
  actions is a dead lever at this scale.
- The next round is **`st-r23`**. The open levers are data-side:
  rejection-sample the model's own VALID oracle-matching actions into the
  flywheel, denser exact-action sim SFT rows, and multi-sentence gold chains
  for chain depth. Keep `npc_grpo.py --action-reward` at its default `off`
  (= the r16 ship recipe).

No tokens or secrets are needed — every download below is anonymous.
Never paste credentials into a notebook (a leaked HF token in a gist is a
documented incident in this project's history).

TPU notes: the first training steps are slow while XLA compiles the step
graph, and generation stages (GRPO rollouts, forge) recompile per sequence
shape — wall-clock is dominated by compile cache warm-up, not a hang.

## Clone and environment

Fresh clone, `uv sync`, then install `torch_xla[tpu]` into the venv — the
lockfile pins a CPU-only torch, and the TPU needs torch + torch_xla +
libtpu at matched versions. `UV_NO_SYNC=1` then protects the provisioned
venv on every later `uv run`.

In [ ]:
import os
if not os.path.exists("traintai"):
    !git clone https://github.com/AnEntrypoint/traintai.git
else:
    !git -C traintai pull --ff-only
%cd traintai

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]
!uv sync
# torch_xla ships no torch dependency: install the matched pair explicitly.
# torch_xla 2.9.0 pairs with torch 2.9.0 (overrides the lock's CPU torch).
!UV_NO_SYNC=1 uv pip install "torch==2.9.0" "torch_xla[tpu]==2.9.0" -f https://storage.googleapis.com/libtpu-releases/index.html

In [ ]:
!UV_NO_SYNC=1 uv run python -c "
import torch_xla.runtime as xr
import torch_xla.core.xla_model as xm
import torch
print('torch', torch.__version__, '| torch_xla runtime:', xr.device_type())
assert xr.device_type() == 'TPU', 'No TPU — Runtime → Change runtime type → TPU (v5e-1)'
print('xla device:', xm.xla_device())
"

## Ship checkpoint

The r16 ship checkpoint is pulled from the GitHub release (anonymous, no
token). Every round starts from it — never from a non-ship round.

In [ ]:
!mkdir -p runs
!curl -sL -o runs/ple-st-r16-grpo.pt https://github.com/AnEntrypoint/traintai/releases/download/v0.1.0/ple-st-r16-grpo.pt
!ls -la runs/

## Run the round

One round = prepare (builds the TinyStories token bins itself on first run,
~300MB anonymous HF download) → 300 SFT steps → 200 GRPO steps → 720-rollout
forge dashboard → held-out sim_eval. Each stage logs to `runs/<tag>*.log` and
a summary block prints at the end. Run ONE round at a time.

In [ ]:
TAG = "st-r23"  # next round; r16 ships, r17–r22 measured non-ship

In [ ]:
!UV_NO_SYNC=1 uv run python src/round.py --prev runs/ple-st-r16-grpo.pt --tag {TAG}

## Holdout generalization gate

Teacher-forced perplexity on the PIPPA holdout (never in the training bins).
Reference points: r16 ship = 358.74, r19 = 357.11. A large jump means the
round overfit the real data — do not ship it.

In [ ]:
!UV_NO_SYNC=1 uv run python src/holdout_eval.py runs/ple-{TAG}-grpo.pt

## Keep the artifacts

Colab runtimes are ephemeral — download the checkpoint and stage logs, then
record the round's measured results in `AGENTS.md` (a result lands once,
with its measurement).

In [ ]:
!tar czf {TAG}-artifacts.tar.gz runs/ple-{TAG}-grpo.pt runs/{TAG}*.log
from google.colab import files
files.download(f"{TAG}-artifacts.tar.gz")